# Shear Center Calculation — S7055 Wing Section

This notebook calculates the shear-center location of the closed S7055 airfoil section using:

1. Thin-walled skin idealised as boom areas.
2. Added stringer/boom areas.
3. Final centroid of the idealised cross-section.
4. Centroidal second moments of area.
5. Basic shear flow due to a vertical shear force.
6. Constant redundant shear flow for a single-cell closed section.
7. Torque about the centroid.
8. Shear-center location.

The calculation uses the existing `week1_sd.py` and `week2a_sd.py` data conventions, but performs the shear-center calculation consistently about the **final centroid**.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rc('font', size=14)
np.set_printoptions(precision=5, suppress=True)

# ---------------------------------------------------------------------
# INPUTS
# ---------------------------------------------------------------------

airfoil_file = "s7055.dat"

# Wing / section data
chord = 0.265
skin_thickness = 0.5e-3

# Stringer geometry
stringer_thickness = 0.5e-3
stringer_length = 20e-3

# Number of stringers to add.
# Change this to match your section.
num_stringers = 8

# Stringer positions as normalized chordwise coordinates (x/c)
#
# Enter upper and lower stringer locations separately.
#
# Example:
# upper_stringer_x_norm = np.array([0.10, 0.30, 0.70, 0.90])
# lower_stringer_x_norm = np.array([0.10, 0.30, 0.70, 0.90])
#
# x/c = 0 -> Leading Edge
# x/c = 1 -> Trailing Edge

upper_stringer_x_norm = np.array([0.16, 0.5, 0.65, 0.78])
lower_stringer_x_norm = np.array([0.16, 0.5, 0.65, 0.78])

# Vertical shear force used only to obtain the shear-center moment.
# The resulting shear-center location is independent of its magnitude.
V = 10000 # N

x_c_cp = 0.3174 # chord ratio not in m 


In [2]:

Xref = 0.3174
Cmref = -0.073
CL = 0.72

Xcp = Xref - Cmref / CL
x_cp = Xcp * chord

print()
print(f"Xcp = {Xcp:.6f} c")
print(f"x_cp = {x_cp:.6e} m")


Xcp = 0.418789 c
x_cp = 1.109791e-01 m


In [3]:

# ---------------------------------------------------------------------
# LOAD AIRFOIL
# ---------------------------------------------------------------------

airfoil = np.loadtxt(airfoil_file) * chord

# Selig coordinates contain a duplicate trailing-edge point.
# Remove the final duplicate if it is coincident with the first point.
if np.allclose(airfoil[0], airfoil[-1]):
    airfoil = airfoil[:-1]

x = airfoil[:, 0]
y = airfoil[:, 1]

n = len(x)

# Closed panels
x_next = np.roll(x, -1)
y_next = np.roll(y, -1)

dx = x_next - x
dy = y_next - y
ds = np.hypot(dx, dy)

# Panel midpoints
x_mid = 0.5 * (x + x_next)
y_mid = 0.5 * (y + y_next)


In [4]:
print(ds)

[0.00051 0.00151 0.00246 0.00334 0.00419 0.005   0.00575 0.00643 0.00705
 0.0076  0.00807 0.00847 0.0088  0.00909 0.00933 0.00951 0.00965 0.00972
 0.00974 0.00973 0.00966 0.00955 0.00939 0.00919 0.00895 0.00867 0.00837
 0.00804 0.00766 0.00727 0.00685 0.0064  0.00595 0.00547 0.00497 0.00447
 0.00396 0.00344 0.00292 0.0024  0.00189 0.00134 0.0012  0.00176 0.00235
 0.00307 0.00389 0.00469 0.00546 0.0062  0.0069  0.00755 0.00814 0.00868
 0.00916 0.00962 0.01001 0.01033 0.01057 0.01075 0.01084 0.01086 0.01079
 0.01067 0.01047 0.0102  0.00987 0.00946 0.00895 0.00838 0.00783 0.00729
 0.00668 0.00599 0.00522 0.00439 0.00349 0.00254 0.00154 0.00052]


In [5]:

# ---------------------------------------------------------------------
# 1. SKIN AREA IDEALISATION
# ---------------------------------------------------------------------

A_skin_panel = skin_thickness * ds
A_skin = np.sum(A_skin_panel)

x_centroid_skin = np.sum(A_skin_panel * x_mid) / A_skin
y_centroid_skin = np.sum(A_skin_panel * y_mid) / A_skin

print(f"Skin area       = {A_skin:.6e} m^2")
print(f"Skin centroid x = {x_centroid_skin:.6e} m")
print(f"Skin centroid y = {y_centroid_skin:.6e} m")

# ---------------------------------------------------------------------
# 2. BOOM AREAS
#
# Use the standard idealisation:
#
# B_i = t/6 * [
#       l_(i-1) (2 + y_(i-1)/y_i)
#     + l_i     (2 + y_(i+1)/y_i)
#     ]
#
# Here the boom formula is evaluated using the skin centroidal
# y-coordinate. The final centroid is then calculated from the
# complete idealised section.
#
# Points extremely close to the reference axis make this formula
# singular. For a robust shear-center calculation, use panel-area
# discretisation instead when such points occur.
# ---------------------------------------------------------------------

yc_skin = y - y_centroid_skin

# Check for points too close to the reference axis.
if np.any(np.abs(yc_skin) < 1e-10):
    raise ValueError(
        "A boom lies essentially on the skin centroidal axis. "
        "The standard boom-area formula becomes singular. "
        "Use a different boom/reference discretisation."
    )

l_prev = np.roll(ds, 1)
l_next = ds

y_prev = np.roll(yc_skin, 1)
y_next = np.roll(yc_skin, -1)

B_skin = (
    skin_thickness / 6.0
    * (
        l_prev * (2.0 + y_prev / yc_skin)
        + l_next * (2.0 + y_next / yc_skin)
    )
)


Skin area       = 2.695333e-04 m^2
Skin centroid x = 1.311096e-01 m
Skin centroid y = 6.145572e-03 m


In [6]:

# ---------------------------------------------------------------------
# 3. ADD STRINGER BOOM AREAS
# ---------------------------------------------------------------------

B_stringer = np.zeros(n)

# Stringer boom area
A_stringer = stringer_thickness * stringer_length


def find_surface_index(x_norm, surface):
    """
    Find the S7055 coordinate closest to the requested x/c
    on either the upper or lower surface.
    """

    x_target = x_norm * chord

    if surface == "upper":
        candidates = np.where(y >= 0)[0]
    elif surface == "lower":
        candidates = np.where(y <= 0)[0]
    else:
        raise ValueError("surface must be 'upper' or 'lower'")

    if len(candidates) == 0:
        raise ValueError(f"No points found on {surface} surface.")

    idx = candidates[np.argmin(np.abs(x[candidates] - x_target))]

    return idx


# -------------------------
# Upper stringers
# -------------------------

upper_stringer_indices = []

for x_norm in upper_stringer_x_norm:

    if not 0 <= x_norm <= 1:
        raise ValueError(
            f"Upper stringer x/c = {x_norm} is outside 0 <= x/c <= 1."
        )

    idx = find_surface_index(x_norm, "upper")

    B_stringer[idx] += A_stringer

    upper_stringer_indices.append(idx)


# -------------------------
# Lower stringers
# -------------------------

lower_stringer_indices = []

for x_norm in lower_stringer_x_norm:

    if not 0 <= x_norm <= 1:
        raise ValueError(
            f"Lower stringer x/c = {x_norm} is outside 0 <= x/c <= 1."
        )

    idx = find_surface_index(x_norm, "lower")

    B_stringer[idx] += A_stringer

    lower_stringer_indices.append(idx)


# Total boom area
B_total = B_skin + B_stringer


In [7]:
print(upper_stringer_x_norm)
print(lower_stringer_x_norm)

print(upper_stringer_indices)
print(lower_stringer_indices)

[0.16 0.5  0.65 0.78]
[0.16 0.5  0.65 0.78]
[30, 20, 16, 12]
[52, 61, 65, 69]


In [8]:

# ---------------------------------------------------------------------
# 4. FINAL CENTROID
# ---------------------------------------------------------------------

A_total = np.sum(B_total)

x_bar = np.sum(B_total * x) / A_total
y_bar = np.sum(B_total * y) / A_total

x_c = x - x_bar
y_c = y - y_bar

print()
print(f"Total idealised area = {A_total:.6e} m^2")
print(f"Final centroid x     = {x_bar:.6e} m")
print(f"Final centroid y     = {y_bar:.6e} m")

# ---------------------------------------------------------------------
# 5. SECOND MOMENTS OF AREA
# ---------------------------------------------------------------------

Ixx = np.sum(B_total * y_c**2)
Iyy = np.sum(B_total * x_c**2)
Ixy = np.sum(B_total * x_c * y_c)

D = Ixx * Iyy - Ixy**2

print()
print(f"Ixx = {Ixx:.6e} m^4")
print(f"Iyy = {Iyy:.6e} m^4")
print(f"Ixy = {Ixy:.6e} m^4")
print(f"D   = {D:.6e} m^8")



Total idealised area = 3.508985e-04 m^2
Final centroid x     = 1.323585e-01 m
Final centroid y     = 6.354864e-03 m

Ixx = 3.908743e-08 m^4
Iyy = 1.942243e-06 m^4
Ixy = -2.562067e-08 m^4
D   = 7.526086e-14 m^8


In [9]:

# ---------------------------------------------------------------------
# 6. BASIC SHEAR FLOW
#
# For a vertical shear force V:
#
# q_b = V/D * (Ixy*x - Ixx*y) integrated as a cumulative
# boom contribution.
#
# The first boom is taken as the cut/reference point.
#
# The panel flow is reconstructed by averaging adjacent nodal flows.
# ---------------------------------------------------------------------

# ================================================================
# 6. BASIC SHEAR FLOW
#    CUT = LAST PANEL (n-1 -> 0)
# ================================================================

te_cut_panel = n - 1

q_b_panel = np.zeros(n)

q_running = 0.0

for i in range(n - 1):

    q_b_panel[i] = q_running

    j = i + 1

    dq = (
        V / D
        * B_total[j]
        * (
            Ixy * x_c[j]
            - Ixx * y_c[j]
        )
    )

    q_running += dq

q_b_panel[te_cut_panel] = 0.0

In [10]:

# ---------------------------------------------------------------------
# 7. REDUNDANT CONSTANT SHEAR FLOW
#
# For a single-cell closed section:
#
# integral(q ds) = 0
#
# Therefore:
#
# q0 = - integral(q_b ds) / integral(ds)
# ---------------------------------------------------------------------



q0 = -np.sum(q_b_panel * ds) / np.sum(ds)

q_panel = q_b_panel + q0

print()
print(f"Basic-flow closure correction q0 = {q0:.6e} N/m")



Basic-flow closure correction q0 = 3.684821e+03 N/m


In [11]:

# ---------------------------------------------------------------------
# 8. TORQUE ABOUT THE FINAL CENTROID
#
# For each straight panel:
#
# dT = q * (x dy - y dx)
#
# For a straight panel:
#
# integral(x dy - y dx)
#     = x_i*y_(i+1) - y_i*x_(i+1)
# ---------------------------------------------------------------------

# ================================================================
# TORQUE ABOUT CENTROID
# ================================================================

x_next = np.roll(x_c, -1)
y_next = np.roll(y_c, -1)

panel_moment_arm = (
    x_c * y_next
    - y_c * x_next
)

T_shear_flow = np.sum(
    q_panel * panel_moment_arm
)

# ================================================================
# SHEAR CENTER
# ================================================================

e = -T_shear_flow / V

x_sc_LE = x_bar + e

print(f"Torque about centroid = {T_shear_flow:.6e} N m")
print(f"Shear-center offset = {e:.6e} m")
print(f"Shear center from LE = {x_sc_LE:.6e} m")
print(f"Shear center x/c = {x_sc_LE/chord:.6f}")


Torque about centroid = -3.960673e+01 N m
Shear-center offset = 3.960673e-03 m
Shear center from LE = 1.363192e-01 m
Shear center x/c = 0.514412


In [12]:

# ================================================================
# TORQUE ABOUT SHEAR CENTER
# ================================================================

T_CP = V * (x_cp - x_sc_LE)

print(f"T_CP = {T_CP:.6e} N m")


# ================================================================
# MEDIAN-LINE AREA
# ================================================================

x_mid = 0.5 * (x + np.roll(x, -1))
y_mid = 0.5 * (y + np.roll(y, -1))

x_mid_next = np.roll(x_mid, -1)
y_mid_next = np.roll(y_mid, -1)

A_m = 0.5 * abs(
    np.sum(
        x_mid * y_mid_next
        - y_mid * x_mid_next
    )
)

print(f"A_m = {A_m:.6e} m^2")


# ================================================================
# TORQUE-INDUCED SHEAR FLOW
# ================================================================

q_CP = T_CP / (2.0 * A_m)

print(f"q_CP = {q_CP:.6e} N/m")


# ================================================================
# FINAL SHEAR FLOW
# ================================================================

q_final = q_panel + q_CP

print()
print(f"Minimum q_final = {np.min(q_final):.6e} N/m")
print(f"Maximum q_final = {np.max(q_final):.6e} N/m")
print(f"Maximum |q_final| = {np.max(np.abs(q_final)):.6e} N/m")

T_CP = -2.534012e+02 N m
A_m = 4.919877e-03 m^2
q_CP = -2.575280e+04 N/m

Minimum q_final = -4.495203e+04 N/m
Maximum q_final = -6.410134e+03 N/m
Maximum |q_final| = 4.495203e+04 N/m


In [13]:
# Author: Mahesh

# Check Critical Buckling Loads

In [14]:
# Position from Wing Root
wing_root_tip_len = 850e-3
aileron_inner_edge = 550e-3     # From Wing Root
rib_spanwise_position = np.array([0, 100, 200, 300, 425, 550, 700, 850])*1e-3

In [15]:
# Finding the Panel Length between Stringers

# print(stringers_spacing)
stringers_idx = np.concatenate((np.flip(upper_stringer_indices), lower_stringer_indices))
stringer_spar_idx = np.insert(stringers_idx, [3, 5], [25, 57])
# print(np.array(range(1, num_stringers)))
# print(stringer_spar_idx)

num_stringer_spar = num_stringers + 2
stringers_spacing = np.zeros(num_stringer_spar + 1)

for i in range(1, num_stringer_spar):
    stringers_spacing[i] = np.sum(ds[:stringer_spar_idx[i] + 1]) - np.sum(ds[:stringer_spar_idx[i-1] + 1])

stringers_spacing[0] = np.sum(ds[:stringer_spar_idx[0] + 1])
stringers_spacing[num_stringer_spar] = np.sum(ds) - np.sum(ds[:stringer_spar_idx[-1] + 1])

In [16]:
# print(stringers_idx)
# print(stringers_spacing)

# print(np.sum(ds) - np.sum(stringers_spacing))

In [17]:
# Wing Parameters
a = rib_spanwise_position[1:] - rib_spanwise_position[:-1]
b = stringers_spacing
t = 5e-4

# Material Aluminum
E = 69e9 
nu = 0.33

Ra = 1
Rb = 1

n_fos = 1.5
n_vn = 3.5
n_fatigue = 1.5
k_stress_conc_factor = 1.5


def Pcr(a_, b_):
    Pcr_array = np.zeros((len(a_), len(b_)))
    for i in range(len(a_)):
        for j in range(len(b_)):
            Kss = 5.34 + 4*(b_[j]/a_[i])**2
            Pcr_array[i, j] = Kss*(np.pi**2*E/(12*(1 - nu**2)))*(t/b_[j])**2*(Ra + (Rb - Ra)*(b_[j]/a_[i])**2/2)
    return Pcr_array/(n_fos*n_vn*n_fatigue*k_stress_conc_factor)

In [18]:
print(Pcr(a, b))

[[2043188.73259 5636646.7917  5307668.50581 3976847.35231 5475278.33261
  1346387.98225 3689924.64645 4428810.79728 4594018.98013 5892416.13472
  4017751.34473]
 [2043188.73259 5636646.7917  5307668.50581 3976847.35231 5475278.33261
  1346387.98225 3689924.64645 4428810.79728 4594018.98013 5892416.13472
  4017751.34473]
 [2043188.73259 5636646.7917  5307668.50581 3976847.35231 5475278.33261
  1346387.98225 3689924.64645 4428810.79728 4594018.98013 5892416.13472
  4017751.34473]
 [1849099.32893 5442557.38804 5113579.10215 3782757.94865 5281188.92895
  1152298.57859 3495835.24279 4234721.39362 4399929.57647 5698326.73106
  3823661.94107]
 [1849099.32893 5442557.38804 5113579.10215 3782757.94865 5281188.92895
  1152298.57859 3495835.24279 4234721.39362 4399929.57647 5698326.73106
  3823661.94107]
 [1743668.04793 5337126.10704 5008147.82115 3677326.66765 5175757.64795
  1046867.29759 3390403.9618  4129290.11262 4294498.29548 5592895.45006
  3718230.66007]
 [1743668.04793 5337126.10704 5008

In [19]:
print(np.min(Pcr(a, b)), np.max(Pcr(a, b)))

1046867.2975875703 5892416.134722008
